### Импорты 

In [ ]:
import os
import json
import gc
import random
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF

from diffusers import StableDiffusionPipeline, ControlNetModel, DDPMScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

import warnings
warnings.filterwarnings("ignore")

torch.set_default_dtype(torch.float32)


### Проверка GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU не найден, будет использоваться CPU ")

### Конфигурация

In [ ]:
class Config:
    dataset_path = "./768/images"          # папка с изображениями
    condition_path = "./768/masks"         # папка с масками Cityscapes
    metadata_file = "./768/metadata_fixed.jsonl"  # файл с описаниями
    output_dir = "./train_lora_only"       # куда сохранять результат
    
    # Размеры и обучение
    resolution = 512
    batch_size = 1
    gradient_accumulation_steps = 4
    num_train_epochs = 20
    
    # LoRA параметры
    lora_rank = 16
    lora_alpha = 16
    lora_dropout = 0.1
    target_modules = ["to_q", "to_k", "to_v", "to_out.0"]
    
    # Оптимизация
    learning_rate = 1e-4
    warmup_steps = 100
    max_grad_norm = 1.0
    
    # Модели 
    controlnet_id = "lllyasviel/sd-controlnet-seg"
    base_model = "runwayml/stable-diffusion-v1-5"

config = Config()
os.makedirs(config.output_dir, exist_ok=True)


### Аугментация (изображение + маска)

In [ ]:
class CityscapesAugmentation:
    """Аугментация пар (изображение, маска)"""
    
    def __init__(self, size=512, p_flip=0.5, max_rotate=10, max_scale=1.15,
                 brightness=0.2, contrast=0.2, saturation=0.2,
                 blur_radius=(0.5, 1.5), noise_std=0.03):
        self.size = size
        self.p_flip = p_flip
        self.max_rotate = max_rotate
        self.max_scale = max_scale
        self.brightness = brightness
        self.contrast = contrast
        self.saturation = saturation
        self.blur_radius = blur_radius
        self.noise_std = noise_std

    def __call__(self, image, mask):
        # 1. Горизонтальное отражение
        if random.random() < self.p_flip:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

        # 2. Масштабирование
        scale = random.uniform(1.0, self.max_scale)
        new_w, new_h = int(image.width * scale), int(image.height * scale)
        image = image.resize((new_w, new_h), resample=Image.BICUBIC)
        mask = mask.resize((new_w, new_h), resample=Image.NEAREST)

        # 3. Поворот
        angle = random.uniform(-self.max_rotate, self.max_rotate)
        image = image.rotate(angle, resample=Image.BICUBIC, expand=False)
        mask = mask.rotate(angle, resample=Image.NEAREST, expand=False)

        # 4. Центральный кадрирование
        left = (image.width - self.size) // 2
        top = (image.height - self.size) // 2
        image = image.crop((left, top, left + self.size, top + self.size))
        mask = mask.crop((left, top, left + self.size, top + self.size))

        # 5. Цветовые преобразования (только изображение)
        if random.random() < 0.5:
            enhancer = ImageEnhance.Brightness(image)
            image = enhancer.enhance(1 + random.uniform(-self.brightness, self.brightness))
        if random.random() < 0.5:
            enhancer = ImageEnhance.Contrast(image)
            image = enhancer.enhance(1 + random.uniform(-self.contrast, self.contrast))
        if random.random() < 0.5:
            enhancer = ImageEnhance.Color(image)
            image = enhancer.enhance(1 + random.uniform(-self.saturation, self.saturation))

        # 6. Лёгкое размытие
        if random.random() < 0.5:
            radius = random.uniform(*self.blur_radius)
            image = image.filter(ImageFilter.GaussianBlur(radius=radius))

        # 7. Гауссов шум
        if random.random() < 0.5:
            img_np = np.array(image).astype(np.float32) / 255.0
            noise = np.random.normal(0, self.noise_std, img_np.shape)
            img_np = np.clip(img_np + noise, 0.0, 1.0)
            image = Image.fromarray((img_np * 255).astype(np.uint8))

        return image, mask


### Датасет

In [ ]:
class CityDataset(Dataset):
    
    def __init__(self, metadata_file, img_dir, cond_dir, size=512):
        self.img_dir = img_dir
        self.cond_dir = cond_dir
        self.samples = []
        
        with open(metadata_file, 'r', encoding='utf-8') as f:
            for line in f:
                self.samples.append(json.loads(line.strip()))
        print(f"📸 Загружено {len(self.samples)} пар")
        
        self.size = size
        self.aug = CityscapesAugmentation(size=size)
        
        # Преобразование для изображения (нормализация в [-1, 1])
        self.img_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        
        # Преобразование для маски (просто в тензор, значения [0, 1])
        self.cond_transform = transforms.Compose([
            transforms.ToTensor()
        ])
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        data = self.samples[idx]
        img_name = data['file_name']
        cond_name = data.get('condition_file', img_name.rsplit('.', 1)[0] + '.png')
        
        img_path = os.path.join(self.img_dir, img_name)
        cond_path = os.path.join(self.cond_dir, cond_name)
        
        img = Image.open(img_path).convert('RGB')
        cond = Image.open(cond_path).convert('RGB')
        
        # Применяем аугментацию
        img, cond = self.aug(img, cond)
        
        return {
            "pixel_values": self.img_transform(img),
            "condition_pixel_values": self.cond_transform(cond),
            "caption": data.get('text', 'a street scene')
        }

### EMA (скользящее среднее)

In [ ]:
class EMA:
    """Экспоненциальное скользящее среднее для весов модели"""
    
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.param_list = [p for p in model.parameters() if p.requires_grad]
        self.register()
        
    def register(self):
        for p in self.param_list:
            self.shadow[id(p)] = p.data.clone()
            
    def update(self):
        for p in self.param_list:
            new_avg = (1.0 - self.decay) * p.data + self.decay * self.shadow[id(p)]
            self.shadow[id(p)] = new_avg.clone()
            
    def apply_shadow(self):
        for p in self.param_list:
            self.backup[id(p)] = p.data
            p.data = self.shadow[id(p)]
            
    def restore(self):
        for p in self.param_list:
            p.data = self.backup.pop(id(p))

### Функция обучения

In [ ]:
def train_lora():
    print("=" * 70)
    print("ЗАПУСК ОБУЧЕНИЯ: ControlNet заморожен, обучается только LoRA")
    print("=" * 70)
    
    # 1. Загрузка ControlNet (предобучен, замораживается)
    print(f"Загрузка ControlNet: {config.controlnet_id}")
    controlnet = ControlNetModel.from_pretrained(
        config.controlnet_id,
        torch_dtype=torch.float32
    ).to(device)
    controlnet.requires_grad_(False)
    controlnet.eval()
    print("ControlNet загружен и заморожен")
    
    # 2. Загрузка Stable Diffusion
    print(f"Загрузка Stable Diffusion: {config.base_model}")
    pipe = StableDiffusionPipeline.from_pretrained(
        config.base_model,
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)
    
    # Оптимизации памяти
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("xformers активен")
    except:
        print("xformers не установлен")
    
    # 3. Замораживаем VAE и текстовый энкодер
    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    
    # 4. Добавляем LoRA в UNet
    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=config.target_modules,
        lora_dropout=config.lora_dropout,
        bias="none"
    )
    pipe.unet = get_peft_model(pipe.unet, lora_config)
    print("\n Обучаемые параметры:")
    pipe.unet.print_trainable_parameters()
    
    # 5. Датасет и загрузчик данных
    dataset = CityDataset(
        config.metadata_file,
        config.dataset_path,
        config.condition_path,
        config.resolution
    )
    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    
    # 6. Оптимизатор (только для параметров LoRA)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, pipe.unet.parameters()),
        lr=config.learning_rate,
        betas=(0.9, 0.999),
        weight_decay=1e-2
    )
    
    # 7. Планировщик скорости обучения
    total_steps = len(dataloader) * config.num_train_epochs // config.gradient_accumulation_steps
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=config.warmup_steps,
        num_training_steps=total_steps
    )
    
    # 8. Шедулер шума
    noise_scheduler = DDPMScheduler.from_pretrained(
        config.base_model,
        subfolder="scheduler"
    )
    
    # 9. EMA для UNet с LoRA
    ema = EMA(pipe.unet, decay=0.999)
    
    # 10. Запуск обучения
    global_step = 0
    best_loss = float('inf')
    
    print(f"\nНАЧАЛО ОБУЧЕНИЯ")
    print(f"   Эпох: {config.num_train_epochs}")
    print(f"   Эффективный размер батча: {config.batch_size * config.gradient_accumulation_steps}")
    print(f"   Всего шагов оптимизации: {total_steps}")
    
    for epoch in range(config.num_train_epochs):
        pipe.unet.train()
        epoch_loss = 0.0
        optimizer.zero_grad()
        
        progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), 
                           desc=f"Эпоха {epoch+1}/{config.num_train_epochs}")
        
        for step, batch in progress_bar:
            # Загружаем данные на устройство
            pixel_values = batch["pixel_values"].to(device, dtype=torch.float32)
            condition_values = batch["condition_pixel_values"].to(device, dtype=torch.float32)
            captions = batch["caption"]
            
            # Кодируем изображение в латентное пространство (без градиентов)
            with torch.no_grad():
                latents = pipe.vae.encode(pixel_values).latent_dist.sample()
                latents = latents * pipe.vae.config.scaling_factor
                
                # Текстовые эмбеддинги
                text_inputs = pipe.tokenizer(
                    captions,
                    padding="max_length",
                    max_length=pipe.tokenizer.model_max_length,
                    truncation=True,
                    return_tensors="pt"
                )
                text_embeddings = pipe.text_encoder(text_inputs.input_ids.to(device))[0]
            
            # Добавляем шум
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            
            # Forward через ControlNet (заморожен, без градиентов)
            with torch.no_grad():
                down_block_res_samples, mid_block_res_sample = controlnet(
                    noisy_latents,
                    timesteps,
                    encoder_hidden_states=text_embeddings,
                    controlnet_cond=condition_values,
                    return_dict=False
                )
            
            # Forward через UNet с LoRA (обучается)
            noise_pred = pipe.unet(
                noisy_latents,
                timesteps,
                text_embeddings,
                down_block_additional_residuals=down_block_res_samples,
                mid_block_additional_residual=mid_block_res_sample
            ).sample
            
            # Вычисляем ошибку
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")
            loss = loss / config.gradient_accumulation_steps
            
            # Обратное распространение
            loss.backward()
            epoch_loss += loss.item() * config.gradient_accumulation_steps
            
            # Шаг оптимизации
            if (step + 1) % config.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, pipe.unet.parameters()),
                    config.max_grad_norm
                )
                optimizer.step()
                scheduler.step()
                ema.update()
                optimizer.zero_grad()
                global_step += 1
            
            # Обновляем прогресс-бар
            progress_bar.set_postfix({
                "loss": f"{loss.item() * config.gradient_accumulation_steps:.5f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })
        
        # Конец эпохи
        avg_loss = epoch_loss / len(dataloader)
        print(f"Эпоха {epoch+1} | Средняя ошибка: {avg_loss:.5f}")
        
        # Сохраняем лучшую модель
        if avg_loss < best_loss and not torch.isnan(torch.tensor(avg_loss)):
            best_loss = avg_loss
            best_dir = os.path.join(config.output_dir, "best")
            os.makedirs(best_dir, exist_ok=True)
            
            ema.apply_shadow()
            pipe.unet.save_pretrained(os.path.join(best_dir, "lora"))
            ema.restore()
            print(f"Сохранена лучшая LoRA (ошибка: {avg_loss:.5f})")
        
        # Очищаем память
        gc.collect()
        torch.cuda.empty_cache()
    
    # Финальное сохранение
    final_dir = os.path.join(config.output_dir, "final")
    os.makedirs(final_dir, exist_ok=True)
    ema.apply_shadow()
    pipe.unet.save_pretrained(os.path.join(final_dir, "lora"))
    ema.restore()
    
    print(f"\n Обучение завершено!")
    print(f"   Лучшая LoRA: {config.output_dir}/best/lora")
    print(f"   Финальная LoRA: {config.output_dir}/final/lora")
    
    return final_dir

print(" Функция обучения готова")

### Функция генерации

In [ ]:
from diffusers import StableDiffusionControlNetPipeline

def generate_images(mask_path, prompt="", lora_path=None, num_images=1):
    """Генерация изображений по маске с использованием обученной LoRA"""
    
    # Загружаем ControlNet
    controlnet = ControlNetModel.from_pretrained(
        config.controlnet_id,
        torch_dtype=torch.float16
    )
    
    # Загружаем базовый пайплайн
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        config.base_model,
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None
    ).to(device)
    
    # Загружаем LoRA (если передана)
    if lora_path:
        pipe.load_lora_weights(lora_path)
        print(f"Загружена LoRA: {lora_path}")
    
    # Включаем оптимизацию памяти
    pipe.enable_xformers_memory_efficient_attention()
    
    # Загружаем маску
    mask = Image.open(mask_path).convert('RGB')
    
    # Генерация
    images = []
    for i in range(num_images):
        result = pipe(
            prompt=prompt,
            image=mask,
            num_inference_steps=30,
            guidance_scale=7.5,
        ).images[0]
        images.append(result)
    
    return images


### Цикл обучения

In [ ]:

if not os.path.exists(config.metadata_file):
    print(f"   - {config.dataset_path} (изображения)")
    print(f"   - {config.condition_path} (маски Cityscapes)")
    print(f"   - {config.metadata_file} (файл с описаниями)")
else:
    model_path = train_lora()
    print(f"\n Готово! Модель сохранена в: {model_path}")